# Coupling RiverBedDynamics and RiverFlowDynamics_HLLC Components

This notebook illustrates the process of running the River Bed Dynamics component, coupled with the shock-capturing **RiverFlowDynamics_HLLC** shallow-water solver, on an initially flat riverbed that receives an excess sediment supply. It then demonstrates how the bed slope is continuously adjusted in both time and space until a new bed elevation equilibrium is reached. For demonstration purposes, we have simulated an event that lasts one day. However, the code can be modified (by altering the commented variables) to extend the simulation until equilibrium is achieved.

This is the HLLC version of the introductory OverlandFlow tutorial. The morphodynamic setup is identical; only the hydraulic driver changes. Because HLLC enters water through a true domain boundary (rather than a rainfall source term), the coupling needs two small adjustments relative to the OverlandFlow version, each flagged in place below: the upstream bed boundary must not overwrite the inlet row, and lateral uniformity is enforced each step to suppress a transverse grid-scale checkerboard.

First, let's import the necessary libraries and modules:

In [ ]:
import copy

import matplotlib.cm as cm
import numpy as np
from IPython.display import clear_output, display
from matplotlib import pyplot as plt

from landlab import imshow_grid
from landlab.components import RiverBedDynamics, RiverFlowDynamics_HLLC
from landlab.grid.mappers import map_mean_of_link_nodes_to_link
from landlab.io.esri_ascii import load as load_esri_ascii

Next, let's define some numerical simulation parameters, time control settings, filenames for accessing the grain size distribution, the digital elevation model (DEM), and set boundary conditions.

In [ ]:
# ASCII raster DEM containing the bed surface elevation
zDEM = "bedElevationDEM.asc"

# ASCII file containing grain size distribution
gsd = np.loadtxt("bed_gsd.txt")

# Elapsed time (sec)
t = 0

# Elapsed time when getting initial conditions (sec)
t0 = 0

# Maximum time step in sec
max_dt = 1

# Maximum simulation time (sec)
sim_max_t = 86400 + max_dt  # To reach eq use sim_max_t = 60 * 86400 + max_dt

# Maximum simulation time when getting initial conditions (sec)
sim_max_t_0 = 86400 + max_dt

# Manning's n
n = 0.03874

# bedload rate at inlet in m3/s
in_qb = -0.0087

# Link Id in which sediment supply enters
in_l = np.array((221, 222))

# Interior nodes of the sediment-supply row (NOT used to inject water in HLLC)
in_n = np.array((129, 130))

# Top-boundary nodes where HLLC injects flow (the hydraulic inlet)
inlet_water_n = np.array((133, 134))

# Inlet flow depth [m] and southward velocity [m/s] imposed by HLLC
entry_h_val = 1.0
entry_v_val = -1.0

# Node ID for fixed Nodes
fixed_nodes_id = np.array((1, 2, 5, 6))

# Node ID for nodes calculated as zero gradient in the bed-elevation boundary condition.
calc_node_id = np.arange(128, 132)

# Nodes Id where the bed elevation will be extracted to check it's evolution and compare it to the analytical solution
sample_node_id = np.arange(5, 126, 4)

# Interval to plot/sample elevation (sec)
save_data_time_interval = 3600

Now we define the variables used to get the analytical solution

In [ ]:
# Flow discharge at each cell (across the face, in m3/s) Negative means towards left
Q = -50

# Critical shear stress for MPM equation
tauCrStar = 0.047

# Exponent in MPM equation
qbStar_exp = 3 / 2

# Coefficient in MPM equation
qbStar_coeff = 8

# Domain initial distance [m] - The DEM may be larger than this, so this indicates the study area.
x0 = 0

# Domain final distance [m] - The DEM may be larger than this, so this indicates the study area.
xf = 1500

Now we create fields and instantiate the RiverFlowDynamics_HLLC component. Let's examine the fields that are mandatory for the RiverFlowDynamics_HLLC component.

In [ ]:
RiverFlowDynamics_HLLC.input_var_names  # Gives the list of all required fields

Therefore, we need to create the surface_water__depth and topographic__elevation fields. Additionally, let's create a copy of the topographic elevation to use for comparison at the end of the simulation.

In [ ]:
# Load the topographic elevation from a DEM and creates the topographic__elevation field
# load_esri_ascii returns a RasterModelGrid with the field already added
with open(zDEM) as f:
    rmg = load_esri_ascii(f, name="topographic__elevation")
z = rmg.at_node["topographic__elevation"]

# Initialize the surface water depth field - Creates the surface_water__depth
rmg.add_zeros("surface_water__depth", at="node")

# A copy of the original topographic__elevation - This will be used in a plot at the end so it is copied as a grid field
rmg["node"]["topographic__elevation_original"] = copy.deepcopy(
    rmg["node"]["topographic__elevation"]
)

It would be beneficial to visualize the geometry at this stage

In [ ]:
imshow_grid(rmg, "topographic__elevation", vmin=0, vmax=45)

Now, let's instantiate the RiverFlowDynamics_HLLC component. Unlike OverlandFlow, which injects water with a rainfall *source term* at the interior nodes `in_n`, HLLC is a dynamic shallow-water solver: flow must enter through a true **domain-boundary** inlet. We therefore pre-wet the channel and inject water at the top-edge channel nodes (`inlet_water_n`), from where it flows downstream through the sediment-supply row and out through the downstream outlet.

In [ ]:
# Pre-wet the channel (below the 45 m walls) to the inlet depth
rmg.at_node["surface_water__depth"][z < 45.0] = entry_h_val

entry_h = np.full(inlet_water_n.size, entry_h_val)
entry_u = np.zeros(inlet_water_n.size)
entry_v = np.full(inlet_water_n.size, entry_v_val)

rfd = RiverFlowDynamics_HLLC(
    rmg,
    mannings_n=n,
    order=2,  # MUSCL: avoids the first-order depth overshoot at dx=50
    fixed_entry_nodes=inlet_water_n,
    entry_nodes_h_values=entry_h,
    entry_nodes_u_values=entry_u,
    entry_nodes_v_values=entry_v,
    wall_edges={"left", "right"},
    update_link_fields=False,  # we write SIGNED link velocities ourselves
)

Now it is time to instantiate the RiverBedDynamics component, to check the required fields we do:

In [ ]:
RiverBedDynamics.input_var_names

We need to create or make sure that these fields exist: 
'surface_water__depth'
'surface_water__velocity'
'topographic__elevation'

Notice that 'topographic__elevation' was already created. In this case, 'surface_water__depth' appears again, but we should verify if it defined in links or nodes. Also, 'surface_water__velocity' has to be created. We can use RiverBedDynamics.var_mapping to check where these variables are mapped.

In [ ]:
RiverBedDynamics.var_mapping

We have 'surface_water__depth' and 'topographic__elevation' at nodes. Therefore, we will map 'surface_water__depth' to links and create 'surface_water__velocity'. For HLLC we seed the link velocity with the signed values mapped from the node solution.

In [ ]:
rmg.add_zeros("surface_water__velocity", at="node")
if "surface_water__velocity" not in rmg.at_link:
    rmg.add_zeros("surface_water__velocity", at="link")
rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg, "surface_water__depth"
)
rmg["link"]["surface_water__velocity"][:] = rfd.map_velocities_to_links()

We set boundaries as closed boundaries, the outlet is set to an open boundary

In [ ]:
rmg.set_watershed_boundary_condition_outlet_id([1, 2], z, 45.0)

Before instantiating RiverBedDynamics, we need to specify for this case the sediment transport and bed elevation boundary conditions. These are optional fields, that is why they are not listed as mandatory. We will use the bed_surface__elevation_fixed_node and sediment_transport__sediment_supply_imposed_link optional fields. First we create these fields.

In [ ]:
# Creates optional fields that will be used in the simulation
# fixed_nodes defines as 1 if a node is fixed or 0 if it can varies in elevation
fixed_nodes = np.zeros_like(z)
fixed_nodes[fixed_nodes_id] = 1  # Assigns 1 to fixed nodes, all the others are zero.

# sediment_transport__sediment_supply_imposed
qb = np.full(rmg.number_of_links, 0.0)
qb[in_l] = in_qb  # Previously defined - # bedload rate at inlet in m3/s

We will pass fixed_nodes and qb during instantiation.

`check_advective_cfl=False` is set here because in a coupled run the timestep is controlled by `RiverFlowDynamics_HLLC`, which manages its own flow-stability CFL. The morphodynamic CFL warning is therefore suppressed to avoid noise during the run. To re-enable it -- for example when running `RiverBedDynamics` standalone -- simply remove this argument (it defaults to `True`).

In [ ]:
rbd = RiverBedDynamics(
    rmg,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_nodes,
    sed_transp__bedload_rate_fix_link=qb,
    check_advective_cfl=False,
)

A little explanation of what we did for the watershed boundary condition. Our DEM represents a long watershed with a given slope in the central region. The outlet is in the south boundary, which according to landlab grid convention is defined as nodes 1 and 2 (that is why we have ...outlet_id([1,2]..., then z contains the bed surface elevation created when we loaded the DEM, and 45 tells the component that if that value exist in the DEM it is an invalid value. Most commonly it is used as -9999.

## Checking the new stability utilities (optional)

RiverBedDynamics provides CFL utilities you can call at any time to understand the stability limits of the morphodynamic solver, independently of the flow solver:

In [ ]:
# CFL stability reference
#
# After the simulation runs, you can call these at any time
# to understand the morphodynamic stability limits:
#
#   dt_adv  = rbd.calc_max_stable_dt_advective(safety=1.0)
#   dt_diff = rbd.calc_max_stable_dt_diffusive(safety=1.0)
#
# Meaningful values require an established flow field, so they
# are printed in the post-run diagnostics cell at the end.
print("CFL utilities are shown in the post-run diagnostics cell.")

Here we define some ghost cells. In this tutorial we are analyzing a 1500 m long reach, but the DEM contains data up to 1650 m. Outside 1500 m, where the sediment supply is defined, the bed elevation is assumed to vary according to a zero gradient condition.

In [ ]:
n_col = rmg.number_of_node_columns
n_row = rmg.number_of_node_rows

n_row_calc_n = int(calc_node_id.shape[0] / n_col)
calc_node_id = np.reshape(calc_node_id, (n_row_calc_n, n_col))

Before starting let's get the analytical solution

In [ ]:
# Initial bed elevation in the study section
z0 = np.reshape(z, (n_row, n_col))[1:-2, 1]
z0 = copy.deepcopy(z0)

# D50 is constant everywhere in this case. Was converted to m
D50 = rbd._bed_surf__median_size_node[0] / 1000

# MPM bedload per unit width - Dimensionless
qbStar = np.abs(in_qb) / (np.sqrt(rbd._R * rbd._g * D50) * D50)

k1 = (np.abs(Q) * n / rmg.dx) ** (3 / 5)
k2 = k1 / (rbd._R * D50)
S = ((1 / k2) * ((qbStar / qbStar_coeff) ** (1 / qbStar_exp) + tauCrStar)) ** (10 / 7)
x = np.linspace(x0, xf, z0.shape[0])
zAnalytic = S * x

Let's run RiverFlowDynamics_HLLC for one day, such that when we use RiverBedDynamics water is already flowing across the landscape. HLLC stores depth and velocity at nodes, so we remap depth and the signed velocity to links every step.

In [ ]:
progress0 = 0
while t0 < sim_max_t_0:
    rfd.run_one_step(dt=max_dt)

    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )
    rmg["link"]["surface_water__velocity"][:] = rfd.map_velocities_to_links()

    t0 += max_dt
    progress = int((t0 / sim_max_t_0) * 100)
    if progress > progress0 + 1:
        print(f"\rGetting initial conditions - Progress: [{progress}%]", end="")
        progress0 = progress

Let's take a look at the topographic conditions and water depth in what will now be time zero.

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

plt.sca(axs[0])
imshow_grid(rmg, "topographic__elevation", vmin=0, vmax=45)
plt.title("Topographic elevation [m]")

plt.sca(axs[1])
imshow_grid(rmg, "surface_water__depth", cmap="Blues")
plt.title("Surface water depth [m]")

plt.tight_layout()
plt.show()

Before executing the main loop, let's define a function that will help us visualize the evolution of the bed elevation and the water surface live as the simulation runs.

In [ ]:
def plotCurrentBedState(z_evolution, wse_current, t):
    """Live view of the coupled simulation.

    Bed-surface elevation (filled) with the water column stacked on top of it,
    the initial bed, faded previous profiles, and the analytical equilibrium
    slope. The elapsed simulated time is annotated in the bottom-right corner.
    """
    zPrevious = z_evolution[1:-1, :]
    zCurrent = z_evolution[-1, :]

    if t >= 86400:
        t_label = f"{t / 86400:.2f} days"
    elif t >= 3600:
        t_label = f"{t / 3600:.2f} hours"
    else:
        t_label = f"{t:.0f} s"

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(11, 6))
    base = min(0.0, float(np.min(zCurrent))) - 1.0

    # Faded history of previous bed profiles
    n_prev = zPrevious.shape[0]
    if n_prev > 0:
        greys = cm.Greys(np.linspace(0.30, 0.65, n_prev))
        for k, row in enumerate(zPrevious):
            ax.plot(
                x,
                row,
                color=greys[k],
                lw=1.0,
                zorder=2,
                label="Previous bed" if k == n_prev - 1 else None,
            )

    # Water column stacked on the current bed
    ax.fill_between(
        x,
        zCurrent,
        wse_current,
        color="#5BA3E0",
        alpha=0.6,
        zorder=3,
        label="Water column",
    )
    ax.plot(x, wse_current, color="#1F6FB8", lw=1.6, zorder=5, label="Water surface")

    # Current bed surface (filled)
    ax.fill_between(x, base, zCurrent, color="#CBB387", zorder=3)
    ax.plot(x, zCurrent, color="#7A5C2E", lw=2.2, zorder=6, label="Bed surface")

    # References
    ax.plot(x, z0, color="black", lw=1.4, zorder=4, label="Initial bed")
    ax.plot(
        x,
        zAnalytic,
        color="#D1495B",
        lw=2.0,
        ls="--",
        zorder=4,
        label="Analytical equilibrium",
    )

    # Elapsed time -- bottom-right corner
    ax.text(
        0.985,
        0.04,
        f"Elapsed time:  {t_label}",
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        ha="right",
        va="bottom",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#888", alpha=0.9),
        zorder=7,
    )

    ax.set_ylabel("Elevation [m]")
    ax.set_xlabel("Streamwise distance [m]   (outlet \u2192 inlet)")
    ax.set_title("Bed and water-surface evolution", fontsize=13, fontweight="bold")
    ax.set_xlim(0, 1500)
    ax.set_ylim(base, 40)
    ax.legend(
        loc="upper left", bbox_to_anchor=(0.0, 0.97), framealpha=0.9, fontsize=9, ncol=2
    )
    ax.grid(alpha=0.25)

    plt.tight_layout()
    display(fig)
    plt.close(fig)

Now, let's execute the loop that drives the simulation for both components.

In [ ]:
save_data_time_interval_original = copy.deepcopy(save_data_time_interval)
z_evolution = rmg["node"]["topographic__elevation"][sample_node_id]
progress0 = 0
while t < sim_max_t:

    # Velocity at previous time -- use current signed link velocity
    rbd._surface_water__velocity_prev_time_link = rmg["link"][
        "surface_water__velocity"
    ].copy()

    rfd.run_one_step(dt=max_dt)

    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )
    rmg["link"]["surface_water__velocity"][:] = rfd.map_velocities_to_links()

    rbd._grid._dt = max_dt
    rbd.run_one_step()

    # Upstream bed boundary condition (Fix 1: row 32 only; inlet row left free)
    for i in calc_node_id:
        rmg["node"]["topographic__elevation"][i] = rmg["node"][
            "topographic__elevation"
        ][i - n_col]

    # Fix 2: enforce lateral uniformity across the two channel columns
    _zg = rmg["node"]["topographic__elevation"].reshape(rmg.shape)
    _lat = 0.5 * (_zg[:, 1] + _zg[:, 2])
    _zg[:, 1] = _lat
    _zg[:, 2] = _lat

    save_data_time_interval = save_data_time_interval - max_dt
    if save_data_time_interval <= 0:
        z_evolution = np.vstack(
            [z_evolution, rmg["node"]["topographic__elevation"][sample_node_id]]
        )
        # Current water-surface elevation along the sampled profile (bed + depth)
        wse_current = (
            rmg["node"]["topographic__elevation"][sample_node_id]
            + rmg["node"]["surface_water__depth"][sample_node_id]
        )
        plotCurrentBedState(z_evolution, wse_current, t)  # live update
        save_data_time_interval = save_data_time_interval_original

    t += max_dt
    progress = int((t / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\rProgress: [{progress}%]", end="")
        progress0 = progress

Let's take a look at the new topography and the water depth field

In [ ]:
z = rmg["node"]["topographic__elevation"]
z0 = rmg["node"]["topographic__elevation_original"]
fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(10, 5))

plt.sca(axs[0])
imshow_grid(rmg, z0, vmin=0, vmax=45, color_for_closed="None")
plt.title("Elev. [m] at t = 0 [d]")

plt.sca(axs[1])
imshow_grid(rmg, z, vmin=0, vmax=45, color_for_closed="None")
plt.title("Elev. [m] at t = 1 [d]")

plt.sca(axs[2])
diff_z = z - z0
imshow_grid(rmg, diff_z, vmin=0, vmax=20.0, color_for_closed="None")
plt.title("Diff. in elev. [m]")  # set a title

plt.sca(axs[3])
imshow_grid(rmg, "surface_water__depth", cmap="Blues", color_for_closed="None")
plt.title("Water depth [m]")

plt.tight_layout()
plt.show()

After one day of simulated time, we can observe changes in the riverbed as the supplied sediment aggrades the reach from the inlet downstream. Extend `sim_max_t` to approach the analytical equilibrium.

---
## Post-Run Diagnostics (New in This Version)

The new component exposes diagnostic attributes you can inspect after (or during) a run
to check numerical health:

In [ ]:
print("=== RiverBedDynamics post-run diagnostics ===")
print()

# GSD normalisation residual -- how far grain fractions drifted from summing to 1
# before the last renormalisation step. Values close to 0 are ideal.
print("GSD normalisation residual (last step):")
print(f"  max  |Sum f_i - 1|  = {rbd._bed_surf__gsd_residual_max:.2e}")
print(f"  mean |Sum f_i - 1|  = {rbd._bed_surf__gsd_residual_mean:.2e}")
print()
if rbd._bed_surf__gsd_residual_max > 1e-3:
    print("  Tip: residual > 1e-3. Consider using gsd_advection_scheme='tvd_minmod'")
    print("  or reducing dt to limit GSD numerical drift.")
else:
    print("  GSD normalisation looks healthy.")
print()

# CFL reference values at the end of the run
dt_adv = rbd.calc_max_stable_dt_advective(safety=1.0)
dt_diff = rbd.calc_max_stable_dt_diffusive(safety=1.0)
print("CFL reference at end of run:")
print(f"  Max stable dt (advective): {dt_adv:.2f} s  (HLLC used {max_dt} s)")
print(f"  Max stable dt (diffusive): {dt_diff:.2f} s")

---
## Optional: Using the Higher-Accuracy RK2 Time Integrator

The new version offers a second-order Runge-Kutta (Heun's method) integrator that
produces significantly smaller truncation errors at the same timestep, at the cost of
two bedload evaluations per step instead of one.

To use it, simply change the `time_stepping` argument at instantiation:

```python
rbd = RiverBedDynamics(
    rmg,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_nodes,
    sed_transp__bedload_rate_fix_link=qb,
    check_advective_cfl=False,
    time_stepping="rk2",   # <-- switch here
)
```

For very long runs (days to months of simulated time) where you want to use a larger
morphodynamic timestep decoupled from the flow solver, the implicit option is available:

```python
    time_stepping="implicit",   # unconditionally stable; uses scipy spsolve
```

The implicit solver is most useful when `RiverBedDynamics` is run in standalone mode
(without coupling to `RiverFlowDynamics_HLLC`) with prescribed hydraulics, allowing dt to be
set to hours or days.

All other parameters and the rest of the script remain identical.